# NEXUS CognitiveTwin — LSTM Training
**Step 3:** Synthetic data generation
**Step 4:** LSTM training (GPU)
Generated data simulates 24hr joint degradation patterns.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

np.random.seed(42)
print('Libraries ready ✅')

## Config

In [ ]:
# ── Constants ──────────────────────────────
HOURS        = 24          # simulation duration
INTERVAL_SEC = 0.5         # 500ms per sample
N_SAMPLES    = int(HOURS * 3600 / INTERVAL_SEC)  # 172,800 rows
N_JOINTS     = 12
N_FEATURES   = 48          # exact LSTM input size

JOINT_NAMES = [
    'FR_hip','FR_thigh','FR_calf',
    'FL_hip','FL_thigh','FL_calf',
    'RR_hip','RR_thigh','RR_calf',
    'RL_hip','RL_thigh','RL_calf',
]

print(f'Total samples : {N_SAMPLES:,}')
print(f'Features/row  : {N_FEATURES}')
print(f'Dataset size  : ~{N_SAMPLES * N_FEATURES * 4 / 1e6:.1f} MB')

## Degradation Pattern Simulator

In [ ]:
def make_degradation_curve(n: int, fail_at: float, mode: str) -> np.ndarray:
    """
    Health curve banao — 100% se gradually degrade hota hai.
    fail_at: fraction of timeline jab failure hogi (0.0-1.0)
    mode: 'linear' | 'exponential' | 'sudden'
    """
    t = np.linspace(0, 1, n)
    if mode == 'linear':
        # Steady wear — bearings, gears
        health = np.where(t < fail_at, 1.0 - 0.6 * (t / fail_at), 0.4 - 0.4 * ((t - fail_at) / (1 - fail_at + 1e-9)))
    elif mode == 'exponential':
        # Accelerating wear — thermal damage
        health = np.exp(-3.0 * (t / fail_at) ** 2)
        health = np.clip(health, 0.05, 1.0)
    else:  # sudden
        # Sharp failure — impact damage
        health = np.where(t < fail_at, 0.95 + np.random.normal(0, 0.02, n), np.random.uniform(0.05, 0.25, n))
    # Add realistic noise
    noise = np.random.normal(0, 0.015, n)
    return np.clip(health + noise, 0.0, 1.0)


# Visualize patterns
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
t = np.linspace(0, 24, 500)
for ax, mode, color in zip(axes, ['linear','exponential','sudden'], ['steelblue','tomato','gold']):
    h = make_degradation_curve(500, 0.7, mode)
    ax.plot(t, h * 100, color=color, lw=2)
    ax.axhline(70, color='orange', ls='--', alpha=0.7, label='Warning (70%)')
    ax.axhline(40, color='red',    ls='--', alpha=0.7, label='Critical (40%)')
    ax.set_title(f'{mode.capitalize()} degradation')
    ax.set_xlabel('Hours'); ax.set_ylabel('Health %')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('degradation_patterns.png', dpi=100)
plt.show()
print('Patterns visualized ✅')

## Generate Training Dataset

In [ ]:
def generate_dataset(n_samples: int) -> pd.DataFrame:
    """
    48 features generate karo:
    - joint_position [12]  : 0-36 rad range, sinusoidal motion
    - joint_velocity [12]  : derived from position
    - joint_effort   [12]  : torque with degradation effect
    - motor_temp     [4]   : per-leg temperature
    - battery_voltage [1]  : slow discharge curve
    - vibration       [1]  : IMU-derived magnitude
    - labels          [12] : 1 = failure within 2hrs, 0 = safe
    """
    # Timestamps
    timestamps = np.arange(n_samples) * 0.5  # seconds

    # ── Assign degradation profile per joint ──
    profiles = [
        (0.85, 'linear'),       # FR_hip  — healthy
        (0.90, 'linear'),       # FR_thigh
        (0.92, 'linear'),       # FR_calf
        (0.88, 'exponential'),  # FL_hip
        (0.75, 'exponential'),  # FL_thigh — moderate wear
        (0.95, 'linear'),       # FL_calf
        (0.60, 'exponential'),  # RR_hip  — FAILING JOINT ⚠️
        (0.80, 'linear'),       # RR_thigh
        (0.88, 'linear'),       # RR_calf
        (0.93, 'sudden'),       # RL_hip
        (0.87, 'linear'),       # RL_thigh
        (0.91, 'exponential'),  # RL_calf
    ]

    # ── Joint health curves ──
    health = np.stack([
        make_degradation_curve(n_samples, fa, mode)
        for fa, mode in profiles
    ], axis=1)  # shape: (N, 12)

    # ── Joint position (sinusoidal walking motion) ──
    t_rad = timestamps * 2 * np.pi / 0.6  # 0.6s gait cycle
    position = np.stack([
        np.sin(t_rad + i * np.pi / 6) * 0.4
        for i in range(12)
    ], axis=1)

    # ── Joint velocity (position derivative) ──
    velocity = np.gradient(position, 0.5, axis=0)

    # ── Joint effort (torque increases as health drops) ──
    base_effort = np.abs(velocity) * 8.0
    effort = base_effort * (2.0 - health) + np.random.normal(0, 0.3, (n_samples, 12))

    # ── Motor temperature (4 legs) ──
    # Temp rises as corresponding joints degrade
    temp = np.stack([
        40 + 35 * (1 - health[:, i*3:(i+1)*3].mean(axis=1)) +
        np.random.normal(0, 1.5, n_samples)
        for i in range(4)
    ], axis=1)  # shape: (N, 4)

    # ── Battery voltage (14.8V → 12.5V over 24hrs) ──
    battery = 14.8 - 2.3 * (timestamps / timestamps[-1]) + np.random.normal(0, 0.05, n_samples)

    # ── Vibration (increases with joint wear) ──
    worst_health = health.min(axis=1)
    vibration = 0.2 + 1.8 * (1 - worst_health) + np.random.normal(0, 0.05, n_samples)

    # ── Labels: failure within 2 hours window ──
    window = int(2 * 3600 / 0.5)  # 2hr in samples
    labels = np.zeros((n_samples, 12), dtype=np.float32)
    for j in range(12):
        for i in range(n_samples):
            future = health[i:min(i+window, n_samples), j]
            if future.min() < 0.40:  # critical threshold
                labels[i, j] = 1.0

    # ── Assemble DataFrame ──
    cols = (
        [f'pos_{j}'   for j in JOINT_NAMES] +
        [f'vel_{j}'   for j in JOINT_NAMES] +
        [f'eff_{j}'   for j in JOINT_NAMES] +
        [f'temp_leg{i}' for i in range(4)] +
        ['battery_v', 'vibration'] +
        [f'label_{j}' for j in JOINT_NAMES]
    )
    data = np.concatenate([
        position, velocity, effort, temp,
        battery.reshape(-1,1), vibration.reshape(-1,1),
        labels
    ], axis=1)

    df = pd.DataFrame(data, columns=cols)
    df.insert(0, 'timestamp', timestamps)
    return df


print('Generating dataset...')
df = generate_dataset(N_SAMPLES)
print(f'Shape  : {df.shape}')
print(f'Memory : {df.memory_usage().sum() / 1e6:.1f} MB')
print(f'\nLabel distribution (RR_hip — failing joint):')
print(f'  Safe    : {(df["label_RR_hip"] == 0).sum():,} samples')
print(f'  Failure : {(df["label_RR_hip"] == 1).sum():,} samples')
df.head(3)

## Save Dataset

In [ ]:
# Save CSV
out_path = 'twin_training_data.csv'
df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Size : {Path(out_path).stat().st_size / 1e6:.1f} MB')

# Quick sanity plot — RR_hip health over 24hrs
hours = df['timestamp'] / 3600
fig, ax = plt.subplots(figsize=(12, 4))
# Reconstruct health from effort (approximate)
ax.plot(hours, 100 - df['eff_RR_hip'].clip(0,80), color='tomato', lw=1.5, label='RR_hip (failing)')
ax.plot(hours, 100 - df['eff_FR_hip'].clip(0,80), color='steelblue', lw=1.5, alpha=0.7, label='FR_hip (healthy)')
ax.axhline(70, color='orange', ls='--', label='Warning 70%')
ax.axhline(40, color='red',    ls='--', label='Critical 40%')
ax.set_xlabel('Hours'); ax.set_ylabel('Approx Health %')
ax.set_title('24hr Joint Health Simulation')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('health_simulation.png', dpi=100)
plt.show()
print('\n✅ Step 3 Complete — Dataset ready for LSTM training!')

---
# Step 4 — LSTM Training
Architecture: Input(24,48) → LSTM(64) → LSTM(32) → Dense(12) sigmoid
Loss: BCELoss | Optimizer: Adam | Epochs: 50

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
import json, time

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── LSTM Config ─────────────────────────────
SEQ_LEN    = 24      # 24 timesteps = 12 seconds of history
N_FEATURES = 48      # input features per timestep
N_JOINTS   = 12      # output — failure prob per joint
BATCH_SIZE = 32
EPOCHS     = 50
LR         = 1e-3
DROPOUT    = 0.2

FEATURE_COLS = (
    [f'pos_{j}' for j in ['FR_hip','FR_thigh','FR_calf','FL_hip','FL_thigh','FL_calf',
                           'RR_hip','RR_thigh','RR_calf','RL_hip','RL_thigh','RL_calf']] +
    [f'vel_{j}' for j in ['FR_hip','FR_thigh','FR_calf','FL_hip','FL_thigh','FL_calf',
                           'RR_hip','RR_thigh','RR_calf','RL_hip','RL_thigh','RL_calf']] +
    [f'eff_{j}' for j in ['FR_hip','FR_thigh','FR_calf','FL_hip','FL_thigh','FL_calf',
                           'RR_hip','RR_thigh','RR_calf','RL_hip','RL_thigh','RL_calf']] +
    [f'temp_leg{i}' for i in range(4)] +
    ['battery_v', 'vibration']
)
LABEL_COLS = [f'label_{j}' for j in
    ['FR_hip','FR_thigh','FR_calf','FL_hip','FL_thigh','FL_calf',
     'RR_hip','RR_thigh','RR_calf','RL_hip','RL_thigh','RL_calf']]

assert len(FEATURE_COLS) == N_FEATURES, f'Expected 48 features, got {len(FEATURE_COLS)}'
print(f'Features : {len(FEATURE_COLS)} ✅')
print(f'Labels   : {len(LABEL_COLS)} ✅')

In [ ]:
class JointDataset(Dataset):
    """
    Sliding window dataset.
    Har sample = last 24 timesteps → next failure probability
    """
    def __init__(self, X: np.ndarray, y: np.ndarray, seq_len: int = SEQ_LEN):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.X) - self.seq_len

    def __getitem__(self, idx):
        # Input: window of seq_len timesteps
        x = self.X[idx : idx + self.seq_len]         # (24, 48)
        # Label: next timestep ke baad ka failure state
        label = self.y[idx + self.seq_len]            # (12,)
        return x, label


# ── Load + preprocess data ──
print('Loading dataset...')
df = pd.read_csv('twin_training_data.csv')

X_raw = df[FEATURE_COLS].values.astype(np.float32)
y_raw = df[LABEL_COLS].values.astype(np.float32)

# Normalize features — LSTM ke liye critical
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Save scaler — inference time pe same scaling chahiye
import pickle
Path('models/lstm_twin').mkdir(parents=True, exist_ok=True)
with open('models/lstm_twin/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Train/val split — time series hai, random shuffle nahi karenge
split = int(len(X_scaled) * 0.85)  # 85% train, 15% val
X_train, X_val = X_scaled[:split], X_scaled[split:]
y_train, y_val = y_raw[:split],    y_raw[split:]

train_ds = JointDataset(X_train, y_train)
val_ds   = JointDataset(X_val,   y_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train samples : {len(train_ds):,}')
print(f'Val samples   : {len(val_ds):,}')
print(f'Batches/epoch : {len(train_loader):,}')
print(f'Input shape   : (batch, {SEQ_LEN}, {N_FEATURES})')

In [ ]:
class CognitiveTwinLSTM(nn.Module):
    """
    NEXUS CognitiveTwin LSTM
    Input  : (batch, 24, 48)
    Layer 1: LSTM(64) + Dropout(0.2)
    Layer 2: LSTM(32) + Dropout(0.2)
    Output : Dense(12) sigmoid → failure prob per joint
    """
    def __init__(self):
        super().__init__()
        self.lstm1   = nn.LSTM(N_FEATURES, 64, batch_first=True)
        self.drop1   = nn.Dropout(DROPOUT)
        self.lstm2   = nn.LSTM(64, 32, batch_first=True)
        self.drop2   = nn.Dropout(DROPOUT)
        self.output  = nn.Linear(32, N_JOINTS)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):              # x: (B, 24, 48)
        out, _ = self.lstm1(x)         # (B, 24, 64)
        out     = self.drop1(out)
        out, _ = self.lstm2(out)       # (B, 24, 32)
        out     = self.drop2(out)
        out     = out[:, -1, :]        # last timestep: (B, 32)
        return self.sigmoid(self.output(out))  # (B, 12)


model     = CognitiveTwinLSTM().to(DEVICE)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal parameters: {total_params:,}')

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def val_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            preds = model(X_batch)
            total_loss += criterion(preds, y_batch).item()
            all_preds.append((preds > 0.7).float())
            all_labels.append(y_batch)
    preds_cat  = torch.cat(all_preds)
    labels_cat = torch.cat(all_labels)
    accuracy   = (preds_cat == labels_cat).float().mean().item()
    return total_loss / len(loader), accuracy


# ── Training ──
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')
print(f'Training on {DEVICE}...\n')

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss          = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc   = val_epoch(model, val_loader, criterion)
    scheduler.step(val_loss)
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    # Best model save
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'models/lstm_twin/model.pt')
        saved = '💾'
    else:
        saved = '  '

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train: {train_loss:.4f} | '
              f'Val: {val_loss:.4f} | '
              f'Acc: {val_acc:.3f} | '
              f'{elapsed:.1f}s {saved}')

print(f'\nBest val loss: {best_val_loss:.4f}')
print('Training complete ✅')

In [ ]:
# ── Loss curve ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs = range(1, EPOCHS + 1)
ax1.plot(epochs, history['train_loss'], label='Train', color='steelblue')
ax1.plot(epochs, history['val_loss'],   label='Val',   color='tomato')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(epochs, history['val_acc'], color='seagreen')
ax2.set_title('Validation Accuracy'); ax2.set_xlabel('Epoch')
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_curve.png', dpi=100)
plt.show()

# ── Save model metadata ──
meta = {
    'architecture': 'CognitiveTwinLSTM',
    'input_shape' : [SEQ_LEN, N_FEATURES],
    'output_shape': N_JOINTS,
    'warn_threshold'    : 0.7,
    'critical_threshold': 0.9,
    'joint_names' : ['FR_hip','FR_thigh','FR_calf','FL_hip','FL_thigh','FL_calf',
                     'RR_hip','RR_thigh','RR_calf','RL_hip','RL_thigh','RL_calf'],
    'feature_cols': FEATURE_COLS,
    'best_val_loss': round(best_val_loss, 6),
    'epochs_trained': EPOCHS,
}
with open('models/lstm_twin/meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Files saved:')
for p in Path('models/lstm_twin').iterdir():
    print(f'  {p} ({p.stat().st_size/1024:.1f} KB)')
print('\n✅ Step 4 Complete — LSTM trained and saved!')
print('Download: models/lstm_twin/model.pt + scaler.pkl + meta.json')